# Workstream 4: Prescriber targeting
**Client question:** which prescribers should a mid-size sales force call on first in the six launch states (TN, KY, IN, OH, MI, WI), and how many sales reps does the launch need?

| Sub-question | Source |
|---|---|
| 4a. Who writes antidepressants, and who writes the competitor brands, by specialty? | CMS Medicare Part D Prescribers by Provider and Drug, 2024 |
| 4b. Which individual prescribers are the best targets? | Same file, scored per prescriber |
| 4c. How concentrated is prescribing (A / B / C target tiers)? | 4b scores |
| 4d. How are the targets spread across the six states? | 4c tiers |
| 4e. How many sales reps, and what does a six-state launch mean for revenue? | 4c tiers + Auvelity launch benchmark (Workstream 2) + Workstream 3 revenue |

**Decisions locked before building**
- **Competitor brands used to find targets:** Vraylar and Rexulti (add-ons: the prescriber already adds a second drug) and Auvelity and Trintellix (branded second-line antidepressants: the prescriber already switches to newer brands). Caplyta is excluded: in 2024 it was approved for bipolar depression, not as a depression add-on.
- **The "plus" check:** the data is one row per prescriber per drug, so it cannot show that one patient takes an antidepressant *plus* an add-on. Instead, a prescriber who writes both antidepressants and add-on brands is treated as already using add-on treatment; add-on brands with few antidepressants suggests schizophrenia or bipolar use.
- **Who counts as a target:** physicians, nurse practitioners and physician assistants (all can prescribe non-controlled drugs in every state).

**Key terms**
- **Prescriber:** anyone who writes prescriptions, identified by a national ID number (NPI).
- **30-day fill:** one month of medicine, so prescribers can be compared fairly.

**Limitations**
- Medicare only: the data is used to *rank* prescribers, not to count total prescriptions. About two-thirds of competitor-brand Medicare fills are for patients under 65 (on disability), but prescribers whose patients are mostly employed adults (commercial insurance) are under-ranked.
- CMS hides rows with fewer than 11 claims, so very small prescribers are missing.
- No diagnosis: the data shows which drug was prescribed, not why. In practice, the ranking would be checked against paid prescription data (e.g. IQVIA Xponent) before deploying reps.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

RAW, OUT, FIG = "../data/raw/", "../data/processed/", "../outputs/"
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)
REGION = ["TN", "KY", "IN", "OH", "MI", "WI"]

## Step 1 (4a): Who writes antidepressants, and who writes the competitor brands?
**Question:** depression is mostly treated in primary care. Is that also true for the competitor brands our drug will compete with, or do a smaller group of specialists write those?

**Method:** sort every prescription in the six states into three groups, then compare each specialty's share of each group.
- **Add-on:** Vraylar, Rexulti.
- **Second-line brand:** Auvelity, Trintellix.
- **Antidepressants:** all other antidepressants (generic and brand). Trazodone is left out because it is mostly prescribed for sleep, and antipsychotics (quetiapine, olanzapine, aripiprazole etc.) are left out because the data cannot say whether they were used for depression.

In [2]:
raw = pd.read_csv(RAW + "cms_partd_provider_drug_DEPRESSION.csv",
                  usecols=["Prscrbr_NPI", "Prscrbr_Last_Org_Name", "Prscrbr_First_Name", "Prscrbr_City",
                           "Prscrbr_State_Abrvtn", "Prscrbr_Type", "Brnd_Name", "Gnrc_Name",
                           "Tot_30day_Fills", "Tot_Benes"])
rx = raw[raw["Prscrbr_State_Abrvtn"].isin(REGION)].copy()

ADDON = ["Vraylar", "Rexulti"]
SECOND_LINE = ["Auvelity", "Trintellix"]
ANTIDEPRESSANTS = [  # generic names; trazodone (mostly used for sleep) and antipsychotics left out
    "Sertraline Hcl", "Escitalopram Oxalate", "Duloxetine Hcl", "Bupropion Hcl", "Bupropion Hbr",
    "Fluoxetine Hcl", "Citalopram Hydrobromide", "Mirtazapine", "Venlafaxine Hcl", "Venlafaxine Besylate",
    "Paroxetine Hcl", "Paroxetine Mesylate", "Desvenlafaxine Succinate", "Desvenlafaxine", "Fluvoxamine Maleate",
    "Vilazodone Hcl", "Levomilnacipran Hcl", "Nefazodone Hcl", "Amitriptyline Hcl", "Nortriptyline Hcl",
    "Doxepin Hcl", "Imipramine Hcl", "Imipramine Pamoate", "Clomipramine Hcl", "Desipramine Hcl",
    "Protriptyline Hcl", "Trimipramine Maleate", "Selegiline Hcl", "Selegiline", "Tranylcypromine Sulfate",
    "Phenelzine Sulfate", "Isocarboxazid"]

def drug_group(row):
    if row["Brnd_Name"] in ADDON:       return "Add-on"
    if row["Brnd_Name"] in SECOND_LINE: return "Second-line brand"
    if row["Gnrc_Name"] in ANTIDEPRESSANTS: return "Antidepressants"
    return None

rx["group"] = rx.apply(drug_group, axis=1)
rx = rx[rx["group"].notna()]

SPECIALTY = {"Psychiatry": "Psychiatrist", "Psychiatry & Neurology": "Psychiatrist", "Neuropsychiatry": "Psychiatrist",
             "Geriatric Psychiatry": "Psychiatrist",
             "Nurse Practitioner": "Nurse practitioner / specialist",
             "Certified Clinical Nurse Specialist": "Nurse practitioner / specialist",  # advanced-practice nurses, often in psychiatry
             "Physician Assistant": "Physician assistant",
             "Family Practice": "Family / general practice", "General Practice": "Family / general practice",
             "Internal Medicine": "Internal medicine"}
rx["specialty"] = rx["Prscrbr_Type"].map(SPECIALTY).fillna("Other")

print(f"Prescribers in the six states: {rx['Prscrbr_NPI'].nunique():,}")
rx.groupby("group")["Tot_30day_Fills"].sum().map("{:,.0f}".format).rename("30-day fills, 2024").to_frame()

Prescribers in the six states: 53,642


,"30-day fills, 2024"
group,
Add-on,"262,978"
Antidepressants,"27,107,201"
Second-line brand,"132,358"


**Each specialty's share of each group** (% of 30-day fills; each column adds to 100%).

In [3]:
ORDER = ["Antidepressants", "Add-on", "Second-line brand"]
fills = rx.pivot_table(index="specialty", columns="group", values="Tot_30day_Fills", aggfunc="sum").fillna(0)[ORDER]
mix = (fills / fills.sum() * 100).round(1).sort_values("Add-on", ascending=False)
mix.to_csv(OUT + "ws4_specialty_mix.csv")
mix

group,Antidepressants,Add-on,Second-line brand
specialty,,,
Nurse practitioner,26.2,54.3,43.2
Psychiatrist,7.3,21.0,27.6
Family / general practice,34.1,10.0,12.1
Internal medicine,21.8,5.8,8.2
Physician assistant,5.2,5.5,5.2
Other,5.4,3.5,3.6


**How many prescribers in each specialty already write a competitor brand, and how much?**

In [4]:
comp = rx[rx["group"] != "Antidepressants"]
people = pd.DataFrame({
    "prescribers": rx.groupby("specialty")["Prscrbr_NPI"].nunique(),
    "write a competitor brand": comp.groupby("specialty")["Prscrbr_NPI"].nunique()})
people["% who write one"] = (people["write a competitor brand"] / people["prescribers"] * 100).round(0)
people["competitor fills per writer"] = (comp.groupby("specialty")["Tot_30day_Fills"].sum()
                                         / people["write a competitor brand"]).round(0)
people = people.sort_values("competitor fills per writer", ascending=False)
people.to_csv(OUT + "ws4_specialty_prescribers.csv")
people

,prescribers,write a competitor brand,% who write one,competitor fills per writer
specialty,,,,
Psychiatrist,2948,1270,43.0,72.0
Nurse practitioner,19209,3690,19.0,54.0
Other,7848,296,4.0,47.0
Physician assistant,3991,480,12.0,44.0
Internal medicine,7096,732,10.0,36.0
Family / general practice,12550,1371,11.0,31.0


### What 4a tells us
**Answer:** depression is treated mostly in primary care, but the competitor brands are written mostly by **nurse practitioners / specialists and psychiatrists**. These two groups are the first-wave targets.

| | Antidepressants | Add-ons | Second-line brands |
|---|---|---|---|
| Nurse practitioners / specialists + psychiatrists | 34% | **77%** | **72%** |
| Family practice + internal medicine | **56%** | 16% | 20% |

- **Primary care starts treatment, but competitor brands are written mainly by specialists.** Family doctors and internists write over half of all antidepressants but few competitor brands. Likely reasons (not provable with this data, which does not follow patients): patients whose first antidepressant fails are referred on; primary care is less comfortable with antipsychotic add-ons (weight and blood-sugar monitoring); specialist practices handle insurer pre-approval routinely.
- **Possible later opportunity in primary care, only if Phase 3 confirms better tolerability** (no monitoring burden). Same trigger as the $1,350 price: if confirmed, raise the price *and* expand to primary care; if not, stay at ~$940 and focus on specialists (Workstream 5).
- **Psychiatrists:** few (2,960), but 43% already write a competitor brand and they write the most each (~72 fills a year). Highest value per visit.
- **Nurse practitioners / specialists:** write over half of add-on fills; 3,761 already write a competitor brand (~55 fills each). The largest group of targets, especially where psychiatrists are scarce.
- **Together, about 5,000 prescribers already write a competitor brand** in these two groups (of ~7,800 in total). This is the starting point for 4b.
- **"Other" is small** (78 specialties, ~2% of competitor fills). Clinical nurse specialists were moved to the nurse group and geriatric psychiatrists to psychiatrists, because they prescribe like them.

**How antidepressants are used from here:** not to choose the target groups, but as a supporting signal in 4b: (1) the "plus" check (a prescriber writing add-on brands *and* antidepressants is treating depression, not schizophrenia), and (2) the "needs convincing" group (high antidepressant volume, no competitor brand yet), a source of later growth.

**Caveat:** the data does not say which nurse practitioners work in psychiatry and which in primary care; 4b can infer this from each one's drug mix.

## Step 2 (4b): Which individual prescribers are the best targets?
**Question:** of the 53,642 prescribers in the six states, which ones should reps visit first?

**Method (decisions agreed):**
1. Build one row per prescriber: competitor-brand fills (Vraylar + Rexulti + Auvelity + Trintellix) and antidepressant fills.
2. Sort each prescriber into one of four groups:

| Group | Rule | Role |
|---|---|---|
| **Already switching** | Writes a competitor brand **and** 100+ antidepressant fills | First-wave targets |
| Likely other use | Writes a competitor brand, under 100 antidepressant fills (probably schizophrenia or bipolar) | Low priority |
| **Needs convincing** | No competitor brand, but in the top 10% for antidepressant fills | Second wave |
| Not targeted | Everyone else | - |

3. Rank "Already switching" by competitor fills, and "Needs convincing" by antidepressant fills.

**Why 100 antidepressant fills:** 100 months of antidepressant a year is at least ~8 patients treated all year. Up to 100, the cut removes only 6.3% of competitor-brand writers (2.4% of their fills); above ~150-200 it starts removing real depression prescribers.

**How to read the table below:** the *band* columns show how many prescribers sit in each range of antidepressant fills (e.g. 100-149). The *cumulative* columns add up all bands from 0, i.e. how many prescribers would be lost in total if the cutoff were set at the top of that band. Below 100, each band holds ~60-125 prescribers (under 1% of competitor fills each); from 100 upward each band holds 260+ and keeps growing, so 100 is where real depression prescribers begin.

In [ ]:
# One row per prescriber
info = (rx.sort_values("Tot_30day_Fills", ascending=False)
          .groupby("Prscrbr_NPI")[["Prscrbr_Last_Org_Name", "Prscrbr_First_Name", "Prscrbr_City",
                                   "Prscrbr_State_Abrvtn", "specialty"]].first())
fills_by = rx.pivot_table(index="Prscrbr_NPI", columns="group", values="Tot_30day_Fills", aggfunc="sum").fillna(0)
pr = info.join(fills_by)
pr["competitor_fills"] = pr["Add-on"] + pr["Second-line brand"]
pr = pr.rename(columns={"Prscrbr_Last_Org_Name": "last_name", "Prscrbr_First_Name": "first_name",
                        "Prscrbr_City": "city", "Prscrbr_State_Abrvtn": "state",
                        "Antidepressants": "antidepressant_fills", "Add-on": "addon_fills",
                        "Second-line brand": "second_line_fills"})
pr = pr.drop(columns=["addon_fills", "second_line_fills"])   # ranking uses the combined competitor fills only

# How the plus-check cutoff changes the list (prescribers who write a competitor brand)
# Band view: how many prescribers sit in each range of antidepressant fills.
# Cumulative view: how many we would lose in total if the cutoff were set at the top of that band.
writers = pr[pr["competitor_fills"] > 0]
EDGES  = [0, 1, 25, 50, 75, 100, 150, 200, 250, 300, 400, 500]
LABELS = ["0", "1-24", "25-49", "50-74", "75-99", "100-149", "150-199", "200-249", "250-299", "300-399", "400-499"]
band = pd.cut(writers["antidepressant_fills"], bins=EDGES, labels=LABELS, right=False)
cut_table = pd.DataFrame({
    "prescribers in band": writers.groupby(band, observed=False).size(),
    "% of competitor writers": writers.groupby(band, observed=False).size() / len(writers) * 100,
    "% of competitor fills": writers.groupby(band, observed=False)["competitor_fills"].sum() / writers["competitor_fills"].sum() * 100})
cut_table["cumulative prescribers"] = cut_table["prescribers in band"].cumsum()
cut_table["cumulative % of writers"] = cut_table["% of competitor writers"].cumsum()
cut_table["cumulative % of fills"] = cut_table["% of competitor fills"].cumsum()
cut_table.index.name = "antidepressant fills"
cut_table.round(1)

In [ ]:
PLUS_CUTOFF = 100
non_writers = pr[pr["competitor_fills"] == 0]
CONVINCE_CUTOFF = non_writers["antidepressant_fills"].quantile(0.90)   # top 10% of prescribers with no competitor brand
print(f"'Needs convincing' cutoff: {CONVINCE_CUTOFF:,.0f}+ antidepressant fills a year")

def target_group(r):
    if r["competitor_fills"] > 0:
        return "Already switching" if r["antidepressant_fills"] >= PLUS_CUTOFF else "Likely other use"
    return "Needs convincing" if r["antidepressant_fills"] >= CONVINCE_CUTOFF else "Not targeted"

pr["target_group"] = pr.apply(target_group, axis=1)

GROUPS = ["Already switching", "Needs convincing", "Likely other use", "Not targeted"]
summary = pr.groupby("target_group").agg(prescribers=("competitor_fills", "size"),
                                         competitor_fills=("competitor_fills", "sum"),
                                         antidepressant_fills=("antidepressant_fills", "sum")).reindex(GROUPS)
summary["% of all prescribers"]       = summary["prescribers"] / summary["prescribers"].sum() * 100
summary["% of competitor fills"]      = summary["competitor_fills"] / summary["competitor_fills"].sum() * 100
summary["% of antidepressant fills"]  = summary["antidepressant_fills"] / summary["antidepressant_fills"].sum() * 100
summary["antidepressant fills per prescriber"] = summary["antidepressant_fills"] / summary["prescribers"]
summary = summary[["prescribers", "% of all prescribers", "competitor_fills", "% of competitor fills",
                   "antidepressant_fills", "% of antidepressant fills", "antidepressant fills per prescriber"]]
summary.style.format({"prescribers": "{:,.0f}", "competitor_fills": "{:,.0f}", "antidepressant_fills": "{:,.0f}",
                      "% of all prescribers": "{:.1f}", "% of competitor fills": "{:.1f}",
                      "% of antidepressant fills": "{:.1f}", "antidepressant fills per prescriber": "{:,.0f}"})

**Who is in each group, by specialty** (number of prescribers).

In [ ]:
by_spec = pd.crosstab(pr["specialty"], pr["target_group"])[GROUPS]
by_spec.loc["Total"] = by_spec.sum()
by_spec.sort_values("Already switching", ascending=False)

**Preview: top 15 "Already switching" prescribers** (ranked by competitor fills).

In [ ]:
pr["rank_in_group"] = None
sw = pr["target_group"] == "Already switching"
nc = pr["target_group"] == "Needs convincing"
pr.loc[sw, "rank_in_group"] = pr.loc[sw, "competitor_fills"].rank(ascending=False, method="first").astype(int)
pr.loc[nc, "rank_in_group"] = pr.loc[nc, "antidepressant_fills"].rank(ascending=False, method="first").astype(int)

COLS = ["first_name", "last_name", "city", "state", "specialty", "competitor_fills",
        "antidepressant_fills"]
top = pr[sw].sort_values("competitor_fills", ascending=False)[COLS].head(15)
top.style.format({"competitor_fills": "{:,.0f}", "antidepressant_fills": "{:,.0f}"})

In [ ]:
pr.sort_values(["target_group", "rank_in_group"]).to_csv(OUT + "ws4_prescriber_scores.csv")
print(f"Saved {len(pr):,} prescribers to ws4_prescriber_scores.csv")

### What 4b tells us
**Answer:** a short, well-defined list. **7,347 "already switching" prescribers (13.7% of all prescribers) write 97.6% of competitor-brand fills.** These are the first-wave targets.

| Group | Prescribers | % of prescribers | % of competitor fills | % of antidepressant fills | Role |
|---|---|---|---|---|---|
| Already switching | 7,347 | 13.7 | 97.6 | 29.1 | **First wave** |
| Needs convincing | 4,581 | 8.5 | 0 | 27.4 | Not a launch target (see below) |
| Likely other use | 492 | 0.9 | 2.4 | 0.1 | Low priority |
| Not targeted | 41,222 | 76.8 | 0 | 43.4 | No rep visits (email, webinars at most) |

- **The "plus" check confirms the method:** only 2.4% of competitor fills come from prescribers who barely treat depression, so in these six states the competitor brands are used overwhelmingly for depression. Ranking on competitor fills is safe.
- **Top 15 confirms 4a:** 13 of the top 15 are psychiatrists (7) or nurse practitioners (6). The #1 prescriber (Detroit, 1,254 competitor fills) is an outlier; #2 to #15 sit within ~500-770. Several top prescribers work in smaller towns (e.g. Cookeville TN, Marion IN, Owosso MI); 4d will classify urban vs rural with official codes and check whether one or two prescribers "own" a town.
- **Primary-care switchers are first wave:** 2,067 family doctors and internists already write competitor brands. They run large depression practices (~1,520 antidepressant fills each, 11.6% of all antidepressant fills in the region) and already make the "what next?" decision themselves, so referral is not a concern. They write competitor brands only lightly (~21 competitor fills each, typically), so most will likely fall in lower tiers in 4c.
- **Needs convincing (4,581, 82% family practice / internal medicine): not a launch target.** They treat a lot of depression (~1,620 antidepressant fills each, more than the switchers) but don't write competitor brands, and the reason is unknown. If they refer these patients to psychiatry, there is no opportunity. If they simply aren't used to add-ons, they would need convincing, and that depends on Phase 3 showing low monitoring burden. Shortage areas (4d) give a first hint (no psychiatrist nearby to refer to); a pre-launch survey of family doctors would confirm it (Workstream 5).
- **Not targeted is large but thin:** 77% of prescribers, ~285 antidepressant fills each, no competitor brands. Not worth rep visits.

**Limitations:** Medicare only (ranking, not totals); rows under 11 claims hidden; the data shows where competitor brands are prescribed, not how much attention competitor reps already give these prescribers.

## Step 3 (4c): How concentrated is prescribing? A / B / C tiers
**Question:** among the 7,347 "already switching" prescribers, does a small group write most of the competitor volume? This tests **H4: a small group of prescribers drives most branded use.**

**Why tiers:** reps' time is limited. Tiers decide how often each prescriber is visited, which in 4e sets the number of reps and the cost.

**Method:** sort the switchers from most to least competitor fills and keep a running total.
- **Tier A:** prescribers until the running total reaches 50% of competitor fills (visited most often).
- **Tier B:** the next 30% (visited less often).
- **Tier C:** the last 20% (occasional visits plus email, webinars and samples).

The cut points come from the data, not chosen by hand.

In [ ]:
COMPETITORS = ADDON + SECOND_LINE
sw_list = pr[pr["target_group"] == "Already switching"].sort_values("competitor_fills", ascending=False).copy()
running = sw_list["competitor_fills"].cumsum() / sw_list["competitor_fills"].sum()
# a prescriber belongs to the tier in which their fills START (so the one who crosses 50% is still Tier A)
sw_list["tier"] = pd.cut(running.shift(fill_value=0), bins=[-1, 0.5, 0.8, 1.01], labels=["A", "B", "C"])

# How many of the four competitor brands each prescriber writes, and how much their favourite brand takes
comp_rows = rx[rx["Brnd_Name"].isin(COMPETITORS)]
brand_fills = comp_rows.groupby(["Prscrbr_NPI", "Brnd_Name"])["Tot_30day_Fills"].sum()
sw_list["brands_written"] = brand_fills.groupby(level=0).size().reindex(sw_list.index)
sw_list["favourite_brand_%"] = brand_fills.groupby(level=0).max().reindex(sw_list.index) / sw_list["competitor_fills"] * 100

tiers = sw_list.groupby("tier", observed=True).agg(
    prescribers=("competitor_fills", "size"),
    competitor_fills=("competitor_fills", "sum"),
    min_fills=("competitor_fills", "min"),
    max_fills=("competitor_fills", "max"),
    brands_written=("brands_written", "mean"),
    favourite_brand=("favourite_brand_%", "median"))
tiers.insert(1, "% of switchers", tiers["prescribers"] / tiers["prescribers"].sum() * 100)
tiers.insert(3, "% of competitor fills", tiers["competitor_fills"] / tiers["competitor_fills"].sum() * 100)
tiers = tiers.rename(columns={"min_fills": "fills each (min)", "max_fills": "fills each (max)",
                              "brands_written": "brands written (avg of 4)", "favourite_brand": "favourite brand % (median)"})
tiers.style.format({"prescribers": "{:,.0f}", "% of switchers": "{:.1f}", "competitor_fills": "{:,.0f}",
                    "% of competitor fills": "{:.1f}", "fills each (min)": "{:,.0f}", "fills each (max)": "{:,.0f}",
                    "brands written (avg of 4)": "{:.1f}", "favourite brand % (median)": "{:.0f}"})

**Who is in each tier, by specialty** (number of prescribers).

In [ ]:
tier_spec = pd.crosstab(sw_list["specialty"], sw_list["tier"])
tier_spec["Total"] = tier_spec.sum(axis=1)
tier_spec.loc["Total"] = tier_spec.sum()
tier_spec.sort_values("A", ascending=False)

**Concentration curve (tests H4).** Each point on the line is one prescriber, ranked from most to least competitor fills; the line shows the running total. A steep start means a few prescribers write a lot. The dashed lines mark the 50% and 80% cut points that define the tiers.

In [ ]:
# Concentration curve (Pareto): prescribers ranked from most to least competitor fills, with the running total
n_A = (sw_list["tier"] == "A").sum()
n_AB = n_A + (sw_list["tier"] == "B").sum()
x = range(1, len(sw_list) + 1)
y = running.values * 100

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axvspan(0, n_A, color="#1f4e79", alpha=0.18, lw=0)
ax.axvspan(n_A, n_AB, color="#1f4e79", alpha=0.09, lw=0)
ax.plot(x, y, color="#1f4e79", lw=2)
for lvl in (50, 80):
    ax.axhline(lvl, color="#888888", lw=0.8, ls="--")
ax.text(n_A / 2, 84, f"Tier A\n{n_A:,} prescribers\n50% of fills", ha="center", fontsize=10, color="#222222")
ax.text((n_A + n_AB) / 2, 84, f"Tier B\n{n_AB - n_A:,} prescribers\n30% of fills", ha="center", fontsize=10, color="#222222")
ax.text((n_AB + len(sw_list)) / 2, 55, f"Tier C\n{len(sw_list) - n_AB:,} prescribers\n20% of fills", ha="center", fontsize=10, color="#222222")
ax.set_xlim(0, len(sw_list)); ax.set_ylim(0, 100)
ax.set_xlabel("Already-switching prescribers, ranked from most to least competitor fills")
ax.set_ylabel("Running total of competitor fills (%)")
ax.set_title("1 in 6 switching prescribers (1,147) write half of all competitor-brand fills", loc="left", fontsize=12)
for s in ("top", "right"): ax.spines[s].set_visible(False)
ax.grid(axis="y", color="#e5e5e5", lw=0.6); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIG + "ws4_concentration_curve.png", dpi=200, bbox_inches="tight")
plt.show()

**Evidence check: do prescribers who already use brands adopt a new brand faster?**
Auvelity is the newest of the four competitor brands (launched October 2022), so the 2024 data shows who picked up a new brand quickly. To compare fairly, only prescribers with similar, large depression practices (1,000+ antidepressant fills a year) are included. They are grouped by how much they use the *other three* brands (Vraylar, Rexulti, Trintellix) as a share of their depression fills, and the table shows what % had started writing Auvelity.

In [ ]:
auvelity_writers = set(rx.loc[rx["Brnd_Name"] == "Auvelity", "Prscrbr_NPI"])
other3 = rx[rx["Brnd_Name"].isin(["Vraylar", "Rexulti", "Trintellix"])].groupby("Prscrbr_NPI")["Tot_30day_Fills"].sum()

big = pr[pr["antidepressant_fills"] >= 1000].copy()
big["other_brand_fills"] = other3.reindex(big.index).fillna(0)
big["other_brand_share_%"] = big["other_brand_fills"] / (big["other_brand_fills"] + big["antidepressant_fills"]) * 100
big["wrote_auvelity"] = big.index.isin(auvelity_writers)
big["brand use"] = pd.cut(big["other_brand_share_%"], bins=[-0.1, 0, 2, 5, 10, 100],
                          labels=["None", "0-2%", "2-5%", "5-10%", "10%+"])

adoption = big.groupby("brand use", observed=True).agg(prescribers=("wrote_auvelity", "size"),
                                                       adopted_auvelity=("wrote_auvelity", "sum"))
adoption["% who adopted Auvelity"] = adoption["adopted_auvelity"] / adoption["prescribers"] * 100
adoption.style.format({"prescribers": "{:,.0f}", "adopted_auvelity": "{:,.0f}", "% who adopted Auvelity": "{:.1f}"})

**Result:** among prescribers with equally large depression practices, those who already use brands were far more likely to adopt the newest one: **37.6% of heavy brand users (10%+) vs 0.3% of those using none** (about 100x). This supports focusing launch effort on prescribers who already write competitor brands (the A/B/C tiers), rather than on large practices that don't use brands yet.

**Caveats:** (1) a pattern, not proof of cause: Axsome's reps likely visited heavy brand users first; (2) prescribers writing Auvelity fewer than 11 times are hidden; (3) one brand, one year, six states.

### What 4c tells us
**Answer: H4 confirmed.** A small group drives most branded use: **1,147 prescribers (about 2% of all 53,642 in the six states, 1 in 6 switchers) write half of all competitor-brand fills.**

| Tier | Prescribers | % of competitor fills | Competitor fills each | Brands written (of 4) | Visits (sized in 4e) |
|---|---|---|---|---|---|
| **A** | 1,147 | 50% | 88 - 1,254 | 2.9 | Most often (e.g. every 2-3 weeks) |
| **B** | 2,075 | 30% | 34 - 88 | 1.9 | Less often (e.g. monthly) |
| **C** | 4,125 | 20% | 11 - 34 | 1.1 | Occasional, plus email, webinars and samples |

- **Tier A is 83% nurse practitioners / specialists and psychiatrists** (636 + 318 of 1,147). The client's best reps should be trained for these two groups, and the launch message built around both: psychiatrists as the core specialist prescribers, nurse practitioners as the largest group (including telehealth practices). Emphasis may differ (clinical depth vs practical points such as tolerability, monitoring and pre-approval support); to be confirmed with market research.
- **Tier A prescribers are not loyal to one brand:** they write almost 3 of the 4 competitor brands, and their favourite brand gets only about half of their competitor fills (median). A new mechanism adds an option for them rather than asking them to abandon a brand.
- **Evidence for focusing on current brand users:** among prescribers with equally large depression practices, those already using brands were about 100x more likely to adopt the last new brand (Auvelity: 37.6% vs 0.3%).
- **Primary-care switchers stay at Tier C level for now:** 72% of the 2,067 family doctors and internists who write competitor brands fall in Tier C. They are stepped up only if Phase 3 shows low monitoring burden (same trigger as the $1,350 price and the primary-care expansion).

**Limitations:** tiers use Medicare fills only; prescribers whose other brands fall under the 11-claim limit may look more single-brand than they are; the Auvelity check is a pattern, not proof of cause (Axsome's reps likely visited heavy brand users first).

## Step 4 (4d): How are the targets spread across the six states?
**Question:** how many Tier A, B and C prescribers does each state have? This sets the rep territories and numbers in 4e.

**Columns:** "% of Tier A" = this state's share of all 1,147 Tier A prescribers in the six states; "% of competitor fills" = this state's share of all competitor fills written by the 7,347 switchers. Each column adds to 100% down the states.

In [ ]:
by_state = pd.crosstab(sw_list["state"], sw_list["tier"])
by_state["Total targets"] = by_state.sum(axis=1)
by_state["% of targets"] = by_state["Total targets"] / by_state["Total targets"].sum() * 100
by_state["% of Tier A"] = by_state["A"] / by_state["A"].sum() * 100
by_state["% of competitor fills"] = sw_list.groupby("state")["competitor_fills"].sum() / sw_list["competitor_fills"].sum() * 100
by_state = by_state.sort_values("A", ascending=False)
by_state.loc["Total"] = by_state.sum()
by_state.to_csv(OUT + "ws4_targets_by_state.csv")
by_state.style.format({"A": "{:,.0f}", "B": "{:,.0f}", "C": "{:,.0f}", "Total targets": "{:,.0f}",
                       "% of targets": "{:.1f}", "% of Tier A": "{:.1f}", "% of competitor fills": "{:.1f}"})

### What 4d tells us
**Answer:** Ohio holds the most targets (30% of Tier A, 26% of competitor fills), followed by Michigan and Indiana (~18-19% each); Tennessee and Kentucky ~11-13% of Tier A; Wisconsin the smallest (~9%).

| State | Tier A | Total targets | % of Tier A | Insurer hurdles (Workstream 3) | Launch implication |
|---|---|---|---|---|---|
| Ohio | 339 | 1,714 | 30% | Easiest | **Start here** |
| Tennessee | 148 | 1,231 | 13% | Easiest | **Next**: most NP-heavy state (58% of targets) |
| Michigan | 219 | 1,415 | 19% | Hardest (pre-approval on 44-56% of plans) | Reps launch together with field reimbursement managers |
| Indiana | 209 | 1,241 | 18% | Moderate | Standard launch |
| Kentucky | 130 | 1,136 | 11% | Hard (pre-approval on antipsychotic add-ons, step therapy on antidepressants) | Field reimbursement managers |
| Wisconsin | 102 | 610 | 9% | Moderate | Standard launch |

- **Start where targets and easy coverage meet:** Ohio first (most Tier A, easiest coverage), then Tennessee (easy coverage, mostly nurse practitioners, the core target group).
- **Michigan: demand plus access.** Many targets but the hardest pre-approval rules. Reps create demand while field reimbursement managers (Workstream 3 package) make sure prescriptions get filled; without them, rep effort is lost to paperwork.
- **One national sales team, not state teams.** Nurse practitioners and psychiatrists are present in every state, so all reps are trained for both. The mix differs (Wisconsin 31% and Michigan 23% psychiatrists: more clinical-depth conversations; Tennessee 58% and Kentucky 55% nurse practitioners), and the tiers already direct rep time to whoever the top prescribers are in each state.

## Step 5 (4e): How many reps, what do they cost, and what does a six-state launch earn?
**Question:** how big a sales force do the A/B/C tiers need, and how does its cost compare with the revenue the six states can bring?

**Method**
1. Visits needed a year = Tier A × 20 (every 2-3 weeks) + Tier B × 12 (monthly) + Tier C × 4 (quarterly, plus email and webinars).
2. Reps = visits needed ÷ visits one rep can make in a year.
3. Cost = reps × fully loaded cost per rep.
4. Six-state revenue = Workstream 3 national revenue (at ~$940/month) × the six states' share of the national branded market.

**Inputs (industry planning benchmarks, not client data, so shown as ranges)**

| Input | Fewest reps | Base | Most reps | Source |
|---|---|---|---|---|
| Visits per rep a year | 1,320 (6 a day) | 1,100 (5 a day) | 880 (4 a day) | 4-6 visits a day × ~220 working days ([Rework, Pharma Field Sales Economics](https://resources.rework.com/libraries/pharmaceutical-growth/pharma-field-sales-economics)) |
| Fully loaded cost per rep a year | $150K | $200K | $250K | Salary (~$90K base, [Payscale](https://www.payscale.com/research/US/Job=Pharmaceutical_Sales_Representatives/Salary)) is typically 40-50% of total cost (benefits, car, travel, systems, training, managers); developed-market estimate $134-219K (Rework); US specialty reps at the higher end |

**Important:** the Workstream 3 revenue (~$299M in year 2 at ~$940) used Auvelity's **national** sales. A six-state launch earns only the six states' share; the national figure applies only if the client later expands nationally (Workstream 5).

In [ ]:
VISITS = {"A": 20, "B": 12, "C": 4}          # visits per prescriber per year, by tier
tier_counts = sw_list["tier"].value_counts().reindex(["A", "B", "C"])
visits_needed = sum(tier_counts[t] * VISITS[t] for t in VISITS)
print(f"Visits needed a year: {visits_needed:,.0f}")

CASES = {"Fewest reps": {"visits_per_rep": 1320, "cost_per_rep": 150_000},
         "Base":        {"visits_per_rep": 1100, "cost_per_rep": 200_000},
         "Most reps":   {"visits_per_rep":  880, "cost_per_rep": 250_000}}
import math
force = pd.DataFrame({name: {"visits per rep a year": c["visits_per_rep"],
                             "reps": math.ceil(visits_needed / c["visits_per_rep"]),   # round up: you can't hire part of a rep
                             "cost per rep ($K)": c["cost_per_rep"] / 1e3,
                             "sales-force cost a year ($M)": math.ceil(visits_needed / c["visits_per_rep"]) * c["cost_per_rep"] / 1e6}
                      for name, c in CASES.items()})
force.style.format("{:,.0f}").format("{:,.1f}", subset=pd.IndexSlice["sales-force cost a year ($M)", :])

**Cross-check against Auvelity:** Axsome had about 260 reps nationally after its first year (Workstream 2). Scaled to the six states' share of the branded market, that gives a comparable rep count.

In [ ]:
# Six states' share of the national branded market (all prescribers in the US file, not just the six states)
national_comp = raw[raw["Brnd_Name"].isin(COMPETITORS)]
in_region = national_comp["Prscrbr_State_Abrvtn"].isin(REGION)
SHARE_BASE = national_comp.loc[in_region, "Tot_30day_Fills"].sum() / national_comp["Tot_30day_Fills"].sum()
auv = national_comp[national_comp["Brnd_Name"] == "Auvelity"]
SHARE_UPSIDE = auv.loc[auv["Prscrbr_State_Abrvtn"].isin(REGION), "Tot_30day_Fills"].sum() / auv["Tot_30day_Fills"].sum()
print(f"Six states' share of national competitor-brand fills: {SHARE_BASE:.1%} (base); Auvelity only: {SHARE_UPSIDE:.1%} (upside)")

AUVELITY_REPS = 260
print(f"Auvelity's 260 national reps x {SHARE_BASE:.1%} = {AUVELITY_REPS * SHARE_BASE:.0f} reps (our base: {force.loc['reps', 'Base']:.0f})")

**Revenue ramp for a six-state launch vs sales-force cost.** National net revenue at ~$940/month comes from Workstream 3 (year 2 low / base / high: $251M / $299M / $333M). Year 1 follows Auvelity's ramp (year-1 net sales were 45% of year 2: $130.1M vs $291.4M).

In [ ]:
NATIONAL_Y2 = 299          # $M, Workstream 3 base case at ~$940/month (Middle-low scenario)
Y1_TO_Y2 = 130.1 / 291.4   # Auvelity: year-1 net sales as a share of year 2

ramp = pd.DataFrame({
    "National (Workstream 3)": [NATIONAL_Y2 * Y1_TO_Y2, NATIONAL_Y2],
    f"Six states, base ({SHARE_BASE:.1%})": [NATIONAL_Y2 * Y1_TO_Y2 * SHARE_BASE, NATIONAL_Y2 * SHARE_BASE],
    f"Six states, upside ({SHARE_UPSIDE:.1%})": [NATIONAL_Y2 * Y1_TO_Y2 * SHARE_UPSIDE, NATIONAL_Y2 * SHARE_UPSIDE]},
    index=["Year 1 net revenue ($M)", "Year 2 net revenue ($M)"])
base_cost = force.loc["sales-force cost a year ($M)", "Base"]
ramp.loc["Year 2 revenue ÷ base sales-force cost"] = ramp.loc["Year 2 net revenue ($M)"] / base_cost
ramp.loc["Year 2 revenue ÷ base sales-force cost", "National (Workstream 3)"] = float("nan")
ramp.to_csv(OUT + "ws4_six_state_revenue.csv")
ramp.style.format("{:,.0f}", na_rep="-").format("{:.1f}x", subset=pd.IndexSlice["Year 2 revenue ÷ base sales-force cost", :], na_rep="-")

### What 4e tells us
**Answer:** the six-state launch needs **about 59 reps (range 49-74), costing about $12M a year (range $7-18M)**. Year 2 net revenue of **about $68M (base) to $86M (upside)** covers the sales force roughly 6-7 times.

| | Fewest reps | Base | Most reps |
|---|---|---|---|
| Reps | 49 | **59** | 74 |
| Sales-force cost a year | $7.4M | **$11.8M** | $18.5M |

| Six-state launch at ~$940/month | Base (22.8% of national branded market) | Upside (28.7%, Auvelity share) |
|---|---|---|
| Year 1 net revenue | ~$31M | ~$38M |
| Year 2 net revenue | **~$68M** | ~$86M |

- **The rep count is robust:** the visit calculation (59) matches Auvelity's ~260 national reps scaled to the six states (59).
- **Revenue must be stated for six states, not nationally:** the Workstream 3 figure (~$299M in year 2) is national. The six states are a larger share of the branded market (22.8%) than of the population (~16%), but a six-state launch earns ~$68-86M in year 2. The national figure applies only after a national expansion (Workstream 5).
- **Year 1 is an investment year:** the sales force alone takes ~40% of year-1 revenue (~$12M of ~$31M), before the hub, field reimbursement managers and marketing. By year 2 revenue covers the sales force ~6 times. A full profit-and-loss model is outside this analysis, so no break-even date is claimed.
- **Revenue is the bigger risk, not cost:** the client controls sales-force cost (how many reps, what pay), but revenue depends on insurers, Phase 3 results and prescriber adoption. The real downside is wider than the share-based range shown here (insurer pushback, the ~6-month commercial coverage block, the Workstream 3 low case of ~$251M nationally, ~$57M for six states).
- **Recommendation (Workstream 5):** start with the base team (~59 reps) and set a checkpoint at 6-9 months: measure the % of Tier A prescribers who have prescribed, new prescriptions per month and pre-approval success in Michigan and Kentucky against an Auvelity-based benchmark. If ahead, add reps or plan national expansion; if behind, find the cause first (access problem: more reimbursement support; demand problem: refocus on Tier A, hold hiring).

**Limitations:** rep productivity and cost are industry planning benchmarks, not client data; reps covering small towns may manage fewer visits (territory planning, Workstream 5); revenue scaling assumes the six states' share of the branded market applies to the client's drug.